# Objectives
Welcome to Lab 7 of Natural Language Processing. This notebook focuses on the **BERTopic topic modeling framework**.

Today's objectives are to:

1. Analyze the **initial results** from BERTopic's default settings:
    1. The clustered **TOPICS** and their representative terms.
    2. The DOCUMENTS and their topic distributions.
3. Based on your analysis, **improve the topics** using some of BERTopic's provided parameters and methods.
4. **Evaluate** and compare your BERTopic model with LDiA using **coherence and diversity**.

# Notebook Setup

## Packages
The amazing BERTopic package home page is [here](https://maartengr.github.io/BERTopic/index.html). BERTopic was created by [Maarten Grootendorst](https://www.maartengrootendorst.com/) with a [paper](https://arxiv.org/abs/2203.05794) published in 2022.

In [1]:
#| code-summary: 'Import packages'
# Dataset packages
from datasets import load_dataset

# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', None) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np

# NLP packages
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from bertopic import BERTopic
from gensim.models import CoherenceModel
import gensim.corpora as corpora

ModuleNotFoundError: No module named 'bertopic'

# Datasets
Choose among the many "Text" dataset options from 🤗 [Hugging Face](https://huggingface.co/datasets?modality=modality:text&sort=trending) by MODIFYING `dataset_path` below. For example:

1. [jonathanli/legal-advice-reddit](https://huggingface.co/datasets/jonathanli/legal-advice-reddit)
2. [SetFit/20_newsgroups](https://huggingface.co/datasets/SetFit/20_newsgroups)
3. [tsterbak/eurovision-lyrics-1956-2023](https://huggingface.co/datasets/tsterbak/eurovision-lyrics-1956-2023)
4. [OsamaBsher/AITA-Reddit-Dataset](https://huggingface.co/datasets/OsamaBsher/AITA-Reddit-Dataset) (270K posts, so we stream a random sample)

In [ ]:
#| code-summary: 'Load dataset: choose among OPTIONS'
#| code-fold: false

# MODIFY: choose dataset
#dataset_path = 'jonathanli/legal-advice-reddit'
#dataset_path = 'SetFit/20_newsgroups'
#dataset_path = 'tsterbak/eurovision-lyrics-1956-2023'
dataset_path = 'OsamaBsher/AITA-Reddit-Dataset'

if dataset_path == 'OsamaBsher/AITA-Reddit-Dataset':
    # 270K posts (~680MB): stream a shuffled sample instead of downloading everything
    AITA_N = 5000 # MODIFY: sample size
    ds = load_dataset(dataset_path, split='train', streaming=True)
    df = pd.DataFrame(list(ds.shuffle(seed=42, buffer_size=10_000).take(AITA_N)))
else:
    ds = load_dataset(dataset_path, split='train')
    df = ds.to_pandas()

if dataset_path == 'jonathanli/legal-advice-reddit':
    df['text'] = df['title'] + " " + df['body'] # combine for more "topic-ness"

if dataset_path == 'OsamaBsher/AITA-Reddit-Dataset':
    title = df['title'].str.replace(r'^\W*(aita|wibta|wita)\b\W*', '', regex=True) # drop "aita:" prefix in every title
    df['text'] = title + " " + df['text'] # combine for more "topic-ness"

docs = df['text'].tolist() # keep only 'text'
len(docs)

## Peek at the AITA slice

The streamed sample is a few thousand posts from the full 270K. Columns: `id`, `title`, `text` (post body; combined with the title above), `verdict`, `comment1`, `comment2`, `score`.

▶️ Run the cell below to view a slice of the data.<br>
🔄 Change `START`/`N` to look at a different slice.

In [ ]:
#| code-summary: 'View a slice of the AITA dataset'
#| code-fold: false
START, N = 0, 10 # MODIFY: slice start and size
print(f'{len(df)} posts; columns: {list(df.columns)}')
print(df['verdict'].value_counts())
df.iloc[START:START+N][['title', 'verdict', 'score', 'text']]

In [ ]:
#| code-summary: 'Down-sample dataset (if needed) to speed up code below'
DATASET_N = 2000
RANDOM = 42
#df = df.sample(n=DATASET_N, random_state=RANDOM) # MODIFY: uncomment to down-sample
docs = df['text'].tolist() # keep only 'text'
len(docs)

# BERTopic Default Topics

By default, BERTopic uses the HDBSCAN clustering algorithm to choose an optimal number of topics for us.

▶️ Run: Let's run BERTopic "out of the box" and see what we get.<br>
👁️ Inspect: Is the **initial number of topics** what you might expect from this dataset? (Consider size, length of texts)<br>
🔄 Modify: Most important is `min_topic_size` [tuning](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#min_topic_size) which you can only set during instantiation. Keep default for now. Modify as the **preferred option** below.

📚 Documentation+ from BERTopic:

- [Quick Start](https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html) (code)
- [Initialization API](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.__init__) (full parameter list)
- [Algorithm](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) (Steps #1-6)
- [Video](https://www.youtube.com/watch?v=uZxQz87lb84) (notebook walk-through and Q&A with Maarten Grootendorst)

⚠️ Warning: `nr_topics` parameter can force K topics post-clustering but is <span style="color: red;">warned against</span> [here](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) <span style="color: red;">and noted as "tricky"</span> [here](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#nr_topics).<br>
📝 Note: Training "wall time" from ~30 seconds (N=2k) to ~2 minutes (N=10k)

In [ ]:
#| code-summary: 'Configure and train BERTopic with (mostly) default settings'
#| code-fold: false

# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size
    
    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),
    
    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Initial number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Initial vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

## Topic Embeddings

By default, the clustering step (#3) uses HDBSCAN to cluster documents together into topics. Each topic can then be represented as a combination of its member documents' embeddings in the original 384-dimension space of its `SentenceTransformer` model from step #1.

▶️ Display the model's`topic_embeddings_` attribute to view the post-clustering topic embeddings.<br>
👁️ Notice the meaning of the dimensions and the **dense** representation.<br>

📚 BERTopic: 1️⃣ [Embeddings](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html) ➜  2️⃣ [Dimensionality Reduction](https://maartengr.github.io/BERTopic/getting_started/dim_reduction/dim_reduction.html) ➜ 3️⃣ [Clustering](https://maartengr.github.io/BERTopic/getting_started/clustering/clustering.html) <br>
⚠️ Check `topic_model.embedding_model.embedding_model.max_seq_length`. The embedding model truncates each document after this number of tokens! One solution could be to split long documents IF that is reasonable for your task.<br>
📝 These embeddings consider word sequence and context using self-attention, hence the "Transformers" in its name!

In [ ]:
#| code-summary: 'Display topic embeddings as output of Step #3: Clustering'
display(topic_model.topic_embeddings_.shape)
pd.DataFrame(
    topic_model.topic_embeddings_,
    index = topic_model.get_topic_info()['Topic'],
).head()

## Topic-Term Matrix

Part of BERTopic's magic is how it uses semantic/contextual embeddings to cluster documents into topics and then separately uses the **human-interpretable** word-based TF-IDF to represent each topic. It does this by:

1. Joining all documents of a topic together into **one long document**.
2. Calculating this **class-based** "c"-TF-IDF vector for each topic.

▶️ Display the model's `c_tf_idf` attribute to view the Topic-Term Matrix as c-TF-IDF weights.<br>
👁️ Inspect:

- Notice the meaning of the dimensions and the **sparse** representation.
- Each topic vector does not sum up to 1 as these are not probabilities!

📚 BERTopic: 4️⃣ [Vectorizers](https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html) ➜ 5️⃣ [c-TF-IDF](https://maartengr.github.io/BERTopic/getting_started/ctfidf/ctfidf.html)<br>

In [ ]:
#| code-summary: 'Display Topic-Term Matrix of cTF-IDF weights as output of step #5'
display(topic_model.c_tf_idf_.shape)
pd.DataFrame(
    topic_model.c_tf_idf_.toarray(), # can limit [:n]
    index = topic_model.get_topic_info()['Topic'], # can limit [:n]
    columns = topic_model.vectorizer_model.get_feature_names_out(),
).head()

In [ ]:
#| code-summary: 'Display a few per-topic row term sums'
topic_model.c_tf_idf_[0:10].sum(axis=1)

## ✈️ Topics Summary

▶️ The `get_topic_info()` method is the best way to get a high-level summary of the topics that BERTopic generated.<br>
👁️ Inspect:

- For topic 0, find one of its **representative terms** in the topic 0 row in the **topic-term matrix** above. It should have a relatively high value.
- Are there representative words that do not help to interpret topics such as very common or rare words? Note any **need for vocabulary normalization** when we re-vectorize below.
  
🔄 View more top topics with `.head(top_n)`

📚 BERTopic [get_topic_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info), [set_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels), [update_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)<br>
📝 Notice **topic -1**. HDBSCAN automatically identifies outliers which can not be clustered with any topic. BERTopic provides some strategies for [Outlier reduction](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html).

In [ ]:
#| code-summary: 'Display the topic summary by document count order'
#| code-fold: false
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # MODIFY: to see more/less topics (omit Representative_Docs)

## 🔎 Per-Topic Word Score
▶️ Below are two ways to analyze a specific topic(s)'s top word scores for more in-depth analysis: bar charts and raw scores.<br>
👁️ Do the word score distributions help make sense of some of the most distinctive/relevant topics? <br>
🔄 Enter the topic indices that you are interested in analyzing.

📚 BERTopic [Visualize Terms](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_terms.html), [visualize_barchart()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_barchart), [get_topic()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic)

In [ ]:
#| code-summary: 'Visualize a list of topics by their word scores'
topic_model.visualize_barchart( # (defaults)
    topics = [0, 1, 2],         # (None), list overrides top_n_topics
    top_n_topics = 8,           # (8)
    n_words = 10,               # (5)
    autoscale = True,           # (False), may not see all words otherwise!
)

In [ ]:
#| code-summary: 'Analyze one topic by its raw word scores'
topic_model.get_topic(0)

## Change Topic Representation
We can change which words are used to represent the topic (steps #4-5) by using various tokenization/normalization options that we are already familiar with from Labs 2-3 using `update_topics()` such as:

- Removing stopwords recommended [here](https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#removing-stop-words)
- Lemmatization ← **via our old friend spaCy!**
- N-grams
- Corpus-level vocabulary control: `max_df` (common words), `min_df` (rare words), `max_features`

🔄 Add your options to the `Count_Vectorizer()` configuration.<br>
▶️ Call `update_topics()` with the new vectorizer and include any other changes provided by its parameters.<br>
👁️ Does the vocabulary size change as expected? Does the new representation make the **topic more interpretable?**<br>

📚 scikit-learn [CountVectorizer()](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), BERTopic [Update Topic Representations](https://maartengr.github.io/BERTopic/getting_started/topicrepresentation/topicrepresentation.html), [update_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)<br>
📝 Notes:

- This involves re-calculating the cTF-IDF weights based on the new representation.
- This does NOT change the actual topic clustering from step #3. Modularity! ✨

In [ ]:
#| code-summary: 'Change topic label names'
#| code-fold: false
topn = 10 # MODIFY?

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, max_df=0.5) # MODIFY: normalization options
topic_model.update_topics(docs, vectorizer_model=vectorizer_model, top_n_words=topn) # MODIFY: n_gram_range?
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}')
topic_model.get_topic_info().iloc[:, :-1].head()

## Change Topic Naming

We can also change how the topics are named.

▶️ We use `set_topic_labels()` to manually assign new topic names. It can also be used with `generate_topic_labels()` to change the name formatting template from `'i_word1_word2_word3_word4'`.<br>
🔄 Update any topic names that would help with later analysis.

📚 BERTopic: [set_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels), [generate_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.generate_topic_labels) (formatting parameters)

In [ ]:
#| code-summary: 'Change topic label names'
#| code-fold: false
#topic_model.set_topic_labels({0: 'Cars', 1: 'Sports', 2: 'Religion'}) # MODIFY: rename to better labels
topic_model.get_topic_info().iloc[:, :-2].head() # check after changes

## 🟣 🔵 🟢 Topic-Topic Relationships

Two useful visualizations can be used in conjunction to look for topic-to-topic relationships, including merging candidates.

▶️ The first is `visualize_topics()` where topics can be mapped to 2-dimensions using UMAP.<br>
👁️ Do some/many topics cluster together? Does the clustering make sense to you?<br>
🔄 Try mapping both embeddings and cTF-IDF weights by toggling the `use_ctfidf` parameter.

📚 BERTopic: [Visualize Topics](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html), [visualize_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics)

In [ ]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics( # (defaults)
    use_ctfidf = False,       # (False) map embeddings / True: map cTF-IDF weights
    top_n_topics = None,      # (None) all
    custom_labels = True,     # (False)
)

The second visualization is even more useful to identify clusters of topics by high **cosine similarity**. 

▶️ Run `visualize_heatmap()` to get the idea of topic-to-topic simiarity.<br>
🔄 Modify:

- Recall the `visualize_topics()` 2d plot above to guess an initial `n_clusters` to cluster similar topics. **This is very useful and highly recommended** for analyzing potential topics to merge.
- Try different values of `n_clusters`
- Try both embeddings and cTF-IDF using `use_ctfidf`. Which one exhibits more distinctive clusters?

👁️ Inspect:

- Is it now clear from the similarity clustering which topics are candidates for merging?
- Or do you need to review some actual documents first? → Skip to **BERTopic Documents**.

📚 BERTopic: [Visualize Topics](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html), [visualize_heatmap()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_heatmap)

In [ ]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap( # (defaults)
    use_ctfidf = False,        # (False) map embeddings / True: map cTF-IDF weights
    #n_clusters = 10,           # (None) <- MODIFY: powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None,       # (None) all
    custom_labels = True,      # (False)
)

# BERTopic Modify Topics
BERTopic provides an informative page on various [Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) strategies.

Each modification below should be followed by running:

1. X1: Re-Check Topic Summary
2. And if effective: X2. Re-Check Topic Visualizations

## 🌟 Set `min_topic_size` (Step #3: HDBSCAN clustering)
This is the preferred method: to **rely on the HDBSCAN clustering algorithm** to find your number of topics. This can be used to **both decrease and increase** the number of topics that the clustering provides.

🔄 Check the topic summary to see what topic sizes (under **Count**) are being created, and modify `min_topic_size` accordingly.

- for less topics, try increasing `min_topic_size`
- for more topics, try decreasing `min_topic_size`

▶️ This needs to be done at BERTopic instantiation, i.e., the first cell here under **BERTopic Default Topics**<br>
👁️ Re-check the topic summary and see how the number of topics and their representations changed.<br>

📚 BERTopic: [Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html), [min_topic_size](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#min_topic_size)

In [ ]:
#| code-summary: 'Re-configure and train BERTopic, modifying `min_topic_size`'
#| code-fold: false

# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size
    
    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),
    
    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

## A. Reduce topics automatically (HDBSCAN on cTF-IDF vectors)

▶️ We can re-use HDBSCAN to automatically reduce topics by clustering the cTF-IDF vectors (post step #5).<br>
👁️ Re-check the topic summary and see how the number of topics and their new representations changed.<br>

📚 BERTopic: [Automatic Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#automatic-topic-reduction), [reduce_topics(nr_topics='auto')](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.reduce_topics)<br>
📝 Notes:

- This may be a good first option if there are too many initial topics for manual merging
- `nr_topics='auto'` can also specified at BERTopic instantiation

In [ ]:
#| code-summary: 'Run HDBSCAN to cluster topics by their cTF-IDF vectors'
#| code-fold: false
topic_model.reduce_topics(docs, nr_topics='auto') # can be run n times until converges
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

## B. Merge known topics (Post-Training on cTF-IDF vectors)

🔄 Create a list (or list of lists) to merge specific topics that you noted from your analysis above.<br>
▶️ Input the list to `merge_topics()`<br>
👁️ Re-check the topic summary and see how the merged topics were combined (check **Count**, **Name**, **Representation**), and all topics re-ordered.<br>

📚 BERTopic: [Manual Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#manual-topic-reduction), [merge_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)

In [ ]:
#| code-summary: 'Merge topics that you specify'
#| code-fold: false
topics_to_merge = [1, 2] # MODIFY: list (or list of lists), should not re-run as numbers are re-arranged
topic_model.merge_topics(docs, topics_to_merge)
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

## <span style="color: red;">C. Reduce topics to fixed number (Post-Training on cTF-IDF vectors)</span>
🔄 Set a fixed number of topics in the `nr_topics` parameter<br>
▶️ Run `reduce_topics()` with this `nr_topics` number.<br>
👁️ Re-check the topic summary and see how the topics were reduced to the fixed number.<br>

📚 BERTopic: [Topic Reduction after Training](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#topic-reduction-after-training), [reduce_topics](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.reduce_topics)<br>
⚠️ Warning: `nr_topics` can force K topics post-clustering but is <span style="color: red;">warned against</span> [here](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) and <span style="color: red;">noted as "tricky"</span> [here](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#nr_topics).

In [ ]:
#| code-summary: 'Reduce topics to a specified number of topics'
#| code-fold: false
nr_topics = 10 # MODIFY: number of topics
topic_model.reduce_topics(docs, nr_topics=nr_topics)
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

## X1. Re-Check Topic Summary
👁️ Check modified topics after each modification step above.<br>
🔄 View more top topics with `.head(top_n)`

📚 BERTopic [get_topic_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)

In [ ]:
#| code-summary: 'Display the topic summary by document count order'
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # omit Representative_Docs

### X2. Re-Check Visualizations
👁️ Check topic visualizations after each effective modification step above.<br>
🔄 Try both embeddings and cTF-IDF using `use_ctfidf`

📚 BERTopic [visualize_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics), [visualize_heatmap()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_heatmap)

In [ ]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics(
    use_ctfidf = False, # default False (by embeddings) / True (by cTF-IDF)
    top_n_topics = None, # default None (all)
    custom_labels = True, # default False
)

In [ ]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap(
    use_ctfidf = False, # default False (by embeddings) / True (by cTF-IDF)
    #n_clusters = 8, # default None <- powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None, # default None
    custom_labels = True, # default False
)

# BERTopic Documents

## Document-Topic Matrix

▶️ Run `approximate_distribution()` to generate a "post-hoc approximation of topic distributions across documents" calculated on the cTF-IDF representations (step #5).<br>
👁️ Inspect:

- Notice the meaning of the dimensions.
- The topic row should all sum up to 1.0.

📚 BERTopic: [Topic Distributions](https://maartengr.github.io/BERTopic/getting_started/distribution/distribution.html), [approximate_distribution()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.approximate_distribution)<br>
⚠️ The outliers topics -1 is not included in the output of `approximate_distribution()`<br>
📝 The model's `probabilities_` attribute can similarly be analyzed and visualized. It represents "an approximation of topic probabilities as used in HDBSCAN (step #3)" and requires `calculate_probabilities = True` during [instantiation](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.__init__).

In [ ]:
#| code-summary: "Display document-topic matrix as topic distributions"
topic_distr, _ = topic_model.approximate_distribution(docs, min_similarity=0)
display(topic_distr.shape)
pd.DataFrame(
    topic_distr, # can limit [:n]
    columns = topic_model.get_topic_info()['Topic'][1:] # distributions do not include topic -1!
).rename_axis('document_id').head()

In [ ]:
#| code-summary: 'Display a few per-document row topic sums'
topic_distr[0:10].sum(axis=1)[:, np.newaxis]

## ✈️ Documents Summary

▶️ The `get_document_info()` method summarizes each document's topic assignment and probability.<br>
👁️ For any document, compare the **Probability** score with the topic **distribution** for that document in the document-topic matrix above.<br>
🔄 View more documents with `.head(top_n)` or `.sample(n)`

📚 BERTopic: [get_document_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_document_info)

In [ ]:
#| code-summary: 'Display the document summary by document number order'
topic_model.get_document_info(docs)[['Topic', 'Name', 'Top_n_words', 'Probability']].head()

## 🔎 Per-Document Topic Distribution

🔄 Set the document index in the two cells below to visualize and read a specific document.<br>
▶️ `visualize_distribution()` shows one document's topic distribution as a bar chart.<br>
👁️ Does this topic distribution make sense for this document?<br>

📚 BERTopic: [Visualize Probabilities or Distribution](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#visualize-probabilities-or-distribution), [visualize_distribution()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_distribution)

In [ ]:
#| code-summary: "Visualize ONE document's topic distribution"
topic_model.visualize_distribution(                # (default)
    probabilities = topic_distr[1],                # MODIFY: document index number
    min_probability = 0.015,                       # (0.015)
    custom_labels = True,                          # (False)
)

In [ ]:
#| code-summary: 'Read the actual document for validation'
docs[1] # MODIFY: document index number

## 🟣 🔵 🟢 Document-Topic Relationships

▶️ `visualize_documents()` displays all of the documents using their original (step #1) embeddings reduced to 2-dimensional space and colored by topic assignment<br>
👁️ Hover over the documents to see their text. Are the topics recognizable when reading each document?<br>

📚 BERTopic: [Visualize documents with Plotly](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#visualize-documents-with-plotly), [visualize_documents()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_documents)<br>
📝 As suggested in the documentation, we pre-reduce the embeddings from `SentenceTransformer` by directly calling `UMAP`.

In [ ]:
#| code-summary: 'Visualize documents in 2D space'
from sentence_transformers import SentenceTransformer
from umap import UMAP

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False) # docs should be a list (not Pandas Series)!
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings) # docs should be a list (not Pandas Series)!

# Topic Modelling Evaluation Metrics

The evaluation of the quality of the resulting topics from topic modelling is inherently subjective and requires **human oversight**. There are however several commonly used metrics that can be part of your evaluation strategy together with human interpretation. A good overview is provided by [Abdelrazek et al. (2023)](https://www.sciencedirect.com/science/article/abs/pii/S0306437922001090). We look at **coherence** and **diversity** here, which were used in the BERTopic [paper](https://arxiv.org/abs/2203.05794) to compare performance against other topic modelling approaches and variations of embedding models within its own flexible framework.

**Coherence** aims to measure semantic **similarity WITHIN** each topic by calculating how often the top N terms in a topic (as word pairs) co-occur in the corpus. See papers by [Newman et al. (2010)](https://aclanthology.org/N10-1012.pdf) and [Syed & Spruit (2017)](https://ieeexplore.ieee.org/abstract/document/8259775).
- Coherence requires each topic's **top N words** and the **corpus** to check for co-occurrence of word pairs along a sliding window of variable size.
- We use **Gensim's** [CoherenceModel](https://radimrehurek.com/gensim/models/coherencemodel.html) implementation.
- There are several measures of coherence. Here, we use Normalized Pointwise Mutual Information (NPMI) by default which varies in the range of [-1, 1]

**Diversity** aims to measure each topic's **uniqueness COMPARED TO** other topics by calculating the percentage of unique words in the top N words of each topic, and is briefly desribed by [Dieng et al. (2020)](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00325/96463/Topic-Modeling-in-Embedding-Spaces).
- Diversity only requires each topic's **top N words** and varies from [0, 1]
- We define a simple function to calculate diversity directly copying code from the [TopicDiversity](https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py) class of the **[OCTIS](https://github.com/MIND-Lab/OCTIS)** package.

In [ ]:
#| code-summary: 'Define evaluation functions to measure coherence and diversity'
def get_diversity_octis(topics, topn=10):
    '''Calculate topic diversity. Code ref: https://github.com/MIND-Lab/OCTIS.'''
    unique_words = set()
    for topic in topics:
        unique_words = unique_words.union(set(topic[:topn]))
    return len(unique_words) / (topn * len(topics))

def evaluate_bertopic(model_string, bertopic_model, analyzer, docs, window_size=None, topn=None, coherence_measure='c_npmi'):
    '''Calculate coherence and diversity for a BERTopic model.'''

    # Get BERTopic topics-terms (topn)
    if topn is None:
        topn = len(topic_model.get_topic(0))
    topics_topterms = [
        [word for word, score in bertopic_model.get_topic(topic_id)[:topn]]
         for topic_id in bertopic_model.get_topics()
         if topic_id != -1
    ]

    # Coherence needs docs tokenized in original order
    texts_as_tokens = [analyzer(doc) for doc in docs]
    dictionary = corpora.Dictionary(texts_as_tokens)

    tc = CoherenceModel(
        topics=topics_topterms,
        texts=texts_as_tokens, dictionary=dictionary,
        window_size=window_size, topn=topn, coherence=coherence_measure
    ).get_coherence()
    
    td = get_diversity_octis(topics_topterms, topn)

    return {'model': model_string, 'k_topics': len(topics_topterms), 'coherence': tc, 'diversity': td}

def evaluate_ldia(model_string, df_ldia_topics_terms, analyzer, docs, window_size=None, topn=None, coherence_measure='c_npmi'):
    '''Calculate coherence and diversity for a Latent Dirichelet Allocation model.'''

    # Get LDiA topics-terms (topn)
    if topn is None:
        topn = len(df_ldia_topics_topterms)
    topics_topterms = []
    topics_i = df_ldia_topics_topterms.columns.get_level_values(0).unique()
    for topic_i in topics_i:
        topics_topterms.append(df_ldia_topics_topterms[topic_i]['term'].tolist())

    # Coherence needs docs tokenized in original order
    texts_as_tokens = [analyzer(doc) for doc in docs]
    dictionary = corpora.Dictionary(texts_as_tokens)
    
    tc = CoherenceModel(
        topics=topics_topterms,
        texts=texts_as_tokens, dictionary=dictionary,
        window_size=window_size, topn=topn, coherence=coherence_measure
    ).get_coherence()
    
    td = get_diversity_octis(topics_topterms, topn)

    return {'model': model_string, 'k_topics': len(topics_topterms), 'coherence': tc, 'diversity': td}

In [ ]:
#| code-summary: 'Define function: create DataFrame of top constituent term/topic columns for each topic/document row'
def create_df_rows_topcols(row_col_matrix, rows_i, cols_i, col_type, topcol_n):
    '''Finds the top scoring terms (columns) from a Topic-Term matrix, and wraps into a DataFrame.'''
    list_df_rows = []
    for idx, row in enumerate(row_col_matrix):
        row_scores = [{col_type: cols_i[i], 'score': row[i]} for i in row.argsort()[:-topcol_n - 1:-1]]
        df_row = pd.DataFrame(row_scores)
        df_row.columns = pd.MultiIndex.from_product([[rows_i[idx]], df_row.columns])
        list_df_rows.append(df_row)

    return pd.concat(list_df_rows, axis=1)

## Compare with an LDiA Topic Model

🔄 Set `k_topics_ldia` for a fair comparison with BERTopic. Should you use the same K or use the optimized K for both?<br>
▶️ The code below condenses the pipeline from Lab 6 for LDiA.<br>
👁️ How do these topics compare to those from BERTopic?<br>

📚 scikit-learn [LatentDirichletAllocation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)<br>
📝 Note that we reuse the CountVectorizer from our BERTopic pipeline for a consistent comparison.

In [ ]:
#| code-summary: 'Run the LDiA flow, condensed from Lab 6'
k_topics_ldia = 14 # MODIFY: vs. BERTopic K topics, equalize or optimize each?

# Topic modelling
X_bow = topic_model.vectorizer_model.transform(docs)
ldia_model = LatentDirichletAllocation(n_components=k_topics_ldia)
ldia_model.fit_transform(X_bow)

# Data wrangling to view top terms per topic
ldia_topics = ldia_model.components_
ldia_topics_norm = ldia_topics / ldia_topics.sum(axis=1)[:, None]
ldia_topics_i = [f'topic_{i}' for i in range(ldia_model.n_components)]
ldia_terms = topic_model.vectorizer_model.get_feature_names_out()
df_ldia_topics_topterms = create_df_rows_topcols(ldia_topics_norm, rows_i=ldia_topics_i, cols_i=ldia_terms, col_type='term', topcol_n=topn)
df_ldia_topics_topterms

## Evaluate: Compare Metrics

▶️ The code below collects evaluation metrics for each model for comparison.<br>
👁️ Are the results surprising or consistent with your expectations?<br>

📝 Notes:

- For a project evaluation, it is common to run each model several times as both BERTopic and LDiA are non-deterministic in nature.
- Here, be aware that we are measuring the coherence of the topics based on the corpus that was used to generate them, a sort of **in-sample check**. If our goal is for the topics to be **generalizable**, then you can run coherence on held-out test data.
- We reuse the same CountVectorizer's configured `build_analyzer()` to rebuild the corpus as tokenized texts, required to calculate coherence.

In [ ]:
#| code-summary: 'Evaluate and compare models'
#| code-fold: false
analyzer = topic_model.vectorizer_model.build_analyzer() # Equalize preprocessing
results = list()
results.append(evaluate_bertopic('BERTopic', topic_model, analyzer, docs))
results.append(evaluate_ldia('LDiA', df_ldia_topics_topterms, analyzer, docs))
pd.DataFrame(results).set_index('model').round(4)